## Self Attention

In [ ]:
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.5.1


In [65]:
import torch
import torch.nn as nn

In [66]:
inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your     (x^1)
        [0.55, 0.87, 0.66],  # journey  (x^2)
        [0.57, 0.85, 0.64],  # starts   (x^3)
        [0.22, 0.58, 0.33],  # with     (x^4)
        [0.77, 0.25, 0.10],  # one      (x^5)
        [0.05, 0.80, 0.55],
    ]  # step     (x^6)
)

In [67]:
d_in = inputs.shape[1]
d_out = 2

In [ ]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec


torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
sa_v2.W_key.weight

Parameter containing:
tensor([[ 0.4058, -0.4704,  0.2368],
        [ 0.2134, -0.2601, -0.5105]], requires_grad=True)

In [69]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
print("queries:\n", queries)
print("keys.T:\n", keys.T)
attn_scores = queries @ keys.T
print("attn_scores:\n", attn_scores)
attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
print("attn_weights:\n", attn_weights)

queries:
 tensor([[ 0.6600, -0.2047],
        [ 0.9091, -0.4471],
        [ 0.8960, -0.4419],
        [ 0.5034, -0.2633],
        [ 0.4088, -0.2232],
        [ 0.6628, -0.3292]], grad_fn=<MmBackward0>)
keys.T:
 tensor([[ 0.3147, -0.0298, -0.0170, -0.1054,  0.2185, -0.2258],
        [-0.4016, -0.4459, -0.4262, -0.2724,  0.0482, -0.4782]],
       grad_fn=<PermuteBackward0>)
attn_scores:
 tensor([[ 0.2899,  0.0716,  0.0760, -0.0138,  0.1344, -0.0511],
        [ 0.4656,  0.1723,  0.1751,  0.0259,  0.1771,  0.0085],
        [ 0.4594,  0.1703,  0.1731,  0.0259,  0.1745,  0.0090],
        [ 0.2642,  0.1024,  0.1036,  0.0186,  0.0973,  0.0122],
        [ 0.2183,  0.0874,  0.0882,  0.0177,  0.0786,  0.0144],
        [ 0.3408,  0.1270,  0.1290,  0.0198,  0.1290,  0.0078]],
       grad_fn=<MmBackward0>)
attn_weights:
 tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
       

In [70]:
sa_v2(inputs)

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)

## Causal Attention

In [75]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [77]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
mask_simple

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

In [78]:
masked_simple = attn_weights * mask_simple
print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


In [ ]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)  # dim collapse
row_sums

tensor([[0.1921],
        [0.3700],
        [0.5357],
        [0.6775],
        [0.8415],
        [1.0000]], grad_fn=<SumBackward1>)

In [88]:
masked_simpl_norm = masked_simple / row_sums

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)

In [105]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
mask

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])

In [111]:
mask.bool()

tensor([[False,  True,  True,  True,  True,  True],
        [False, False,  True,  True,  True,  True],
        [False, False, False,  True,  True,  True],
        [False, False, False, False,  True,  True],
        [False, False, False, False, False,  True],
        [False, False, False, False, False, False]])

In [108]:
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
masked

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)

In [115]:
attn_weights = torch.softmax(masked / masked.shape[-1] ** 0.5, dim=-1)
attn_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5299, 0.4701, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3599, 0.3199, 0.3202, 0.0000, 0.0000, 0.0000],
        [0.2647, 0.2478, 0.2479, 0.2395, 0.0000, 0.0000],
        [0.2100, 0.1991, 0.1991, 0.1935, 0.1983, 0.0000],
        [0.1818, 0.1666, 0.1667, 0.1595, 0.1667, 0.1587]],
       grad_fn=<SoftmaxBackward0>)

In [127]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6, 6)

dropout(example)

# PyTorch uses inverted dropout:
# it zeroes a fraction p of entries and multiplies the survivors by 1/(1-p) so the expected value stays the same.

tensor([[2., 2., 2., 2., 2., 2.],
        [0., 2., 0., 0., 0., 0.],
        [0., 0., 2., 0., 2., 0.],
        [2., 2., 0., 0., 0., 2.],
        [2., 0., 0., 0., 0., 2.],
        [0., 2., 0., 0., 0., 0.]])

In [129]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.9402, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.6404, 0.0000, 0.0000, 0.0000],
        [0.5295, 0.4956, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4200, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3332, 0.0000, 0.0000, 0.0000, 0.0000]],
       grad_fn=<MulBackward0>)


In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )  # New

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        # New, _ ops are in-place
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / attn_scores.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

`register_buffer(name, tensor)` registers a tensor as part of the module's **state** (like a parameter, but without gradients).

Key effects:
- **Moved with `.to(device)`** — `mask` automatically follows `model.cuda()`/`model.to("cpu")`, unlike a plain `self.mask = tensor`.
- **Saved/loaded in `state_dict()`** — so it's part of checkpoints, unlike plain attributes.
- **Not trainable** — no gradients, not in `model.parameters()`, so optimizers ignore it.
- **Shows in `model.buffers()`** and can be persistent/non-persistent (`persistent=False` to keep it out of the state_dict).


In [182]:
t1 = torch.rand(8, 4, 10)
t1

tensor([[[0.3821, 0.6605, 0.8536, 0.5932, 0.6367, 0.9826, 0.2745, 0.6584,
          0.2775, 0.8573],
         [0.8993, 0.0390, 0.9268, 0.7388, 0.7179, 0.7058, 0.9156, 0.4340,
          0.0772, 0.3565],
         [0.1479, 0.5331, 0.4066, 0.2318, 0.4545, 0.9737, 0.4606, 0.5159,
          0.4220, 0.5786],
         [0.9455, 0.8057, 0.6775, 0.6087, 0.6179, 0.6932, 0.4354, 0.0353,
          0.1908, 0.9268]],

        [[0.5299, 0.0950, 0.5789, 0.9131, 0.0275, 0.1634, 0.3009, 0.5201,
          0.3834, 0.4451],
         [0.0126, 0.7341, 0.9389, 0.8056, 0.1459, 0.0969, 0.7076, 0.5112,
          0.7050, 0.0114],
         [0.4702, 0.8526, 0.7320, 0.5183, 0.5983, 0.4527, 0.2251, 0.3111,
          0.1955, 0.9153],
         [0.7751, 0.6749, 0.1166, 0.8858, 0.6568, 0.8459, 0.3033, 0.6060,
          0.9882, 0.8363]],

        [[0.9010, 0.3950, 0.8809, 0.1084, 0.5432, 0.2185, 0.3834, 0.3720,
          0.5374, 0.9551],
         [0.7475, 0.4979, 0.8549, 0.2438, 0.7577, 0.4536, 0.4130, 0.5585,
          0.1

In [183]:
t1[:2, :3, :5]

tensor([[[0.3821, 0.6605, 0.8536, 0.5932, 0.6367],
         [0.8993, 0.0390, 0.9268, 0.7388, 0.7179],
         [0.1479, 0.5331, 0.4066, 0.2318, 0.4545]],

        [[0.5299, 0.0950, 0.5789, 0.9131, 0.0275],
         [0.0126, 0.7341, 0.9389, 0.8056, 0.1459],
         [0.4702, 0.8526, 0.7320, 0.5183, 0.5983]]])

In [184]:
batch = torch.stack((inputs, inputs), dim=0)
batch.shape

torch.Size([2, 6, 3])

In [186]:
torch.manual_seed(123)

context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)

context_vec = ca(batch)
context_vec

tensor([[[-0.4519,  0.2216],
         [-0.5856,  0.0087],
         [-0.6284, -0.0607],
         [-0.5664, -0.0832],
         [-0.5511, -0.0978],
         [-0.5290, -0.1073]],

        [[-0.4519,  0.2216],
         [-0.5856,  0.0087],
         [-0.6284, -0.0607],
         [-0.5664, -0.0832],
         [-0.5511, -0.0978],
         [-0.5290, -0.1073]]], grad_fn=<UnsafeViewBackward0>)

&nbsp;
## Simple Multi-head Attention

In [230]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [
                CausalAttention(d_in, d_out, context_length, qkv_bias)
                for _ in range(num_heads)
            ]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


torch.manual_seed(123)

context_length = batch.shape[1]
d_in, d_out = 3, 2

mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs, "\n", context_vecs.shape)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5856,  0.0087,  0.5840,  0.3158],
         [-0.6284, -0.0607,  0.6161,  0.3779],
         [-0.5664, -0.0832,  0.5468,  0.3557],
         [-0.5511, -0.0978,  0.5317,  0.3416],
         [-0.5290, -0.1073,  0.5072,  0.3465]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5856,  0.0087,  0.5840,  0.3158],
         [-0.6284, -0.0607,  0.6161,  0.3779],
         [-0.5664, -0.0832,  0.5468,  0.3557],
         [-0.5511, -0.0978,  0.5317,  0.3416],
         [-0.5290, -0.1073,  0.5072,  0.3465]]], grad_fn=<CatBackward0>) 
 torch.Size([2, 6, 4])


In [231]:
# demonstrate torch.cat
torch.manual_seed(123)
list = [torch.rand(4, 3) for _ in range(4)]
print(list)
print(torch.cat(list))

[tensor([[0.2961, 0.5166, 0.2517],
        [0.6886, 0.0740, 0.8665],
        [0.1366, 0.1025, 0.1841],
        [0.7264, 0.3153, 0.6871]]), tensor([[0.0756, 0.1966, 0.3164],
        [0.4017, 0.1186, 0.8274],
        [0.3821, 0.6605, 0.8536],
        [0.5932, 0.6367, 0.9826]]), tensor([[0.2745, 0.6584, 0.2775],
        [0.8573, 0.8993, 0.0390],
        [0.9268, 0.7388, 0.7179],
        [0.7058, 0.9156, 0.4340]]), tensor([[0.0772, 0.3565, 0.1479],
        [0.5331, 0.4066, 0.2318],
        [0.4545, 0.9737, 0.4606],
        [0.5159, 0.4220, 0.5786]])]
tensor([[0.2961, 0.5166, 0.2517],
        [0.6886, 0.0740, 0.8665],
        [0.1366, 0.1025, 0.1841],
        [0.7264, 0.3153, 0.6871],
        [0.0756, 0.1966, 0.3164],
        [0.4017, 0.1186, 0.8274],
        [0.3821, 0.6605, 0.8536],
        [0.5932, 0.6367, 0.9826],
        [0.2745, 0.6584, 0.2775],
        [0.8573, 0.8993, 0.0390],
        [0.9268, 0.7388, 0.7179],
        [0.7058, 0.9156, 0.4340],
        [0.0772, 0.3565, 0.1479],
     

&nbsp;
## Multi-head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = (
            d_out // num_heads
        )  # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(
            d_out, d_out
        )  # optional: Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        keys = self.W_key(x)  # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(-2, -1)  # Dot product for each head

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec


torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 4
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[ 0.1184,  0.3120, -0.0847, -0.5774],
         [ 0.0300,  0.3238, -0.0771, -0.4415],
         [-0.0038,  0.3272, -0.0742, -0.3892],
         [-0.0076,  0.3158, -0.0715, -0.3705],
         [-0.0101,  0.2981, -0.0702, -0.3581],
         [-0.0096,  0.3014, -0.0695, -0.3566]],

        [[ 0.1184,  0.3120, -0.0847, -0.5774],
         [ 0.0300,  0.3238, -0.0771, -0.4415],
         [-0.0038,  0.3272, -0.0742, -0.3892],
         [-0.0076,  0.3158, -0.0715, -0.3705],
         [-0.0101,  0.2981, -0.0702, -0.3581],
         [-0.0096,  0.3014, -0.0695, -0.3566]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


### Demonstrate torch.cat

In [281]:
torch.manual_seed(123)
list = [torch.rand(4, 3) for _ in range(4)]
print(list)
print(torch.cat(list))

[tensor([[0.2961, 0.5166, 0.2517],
        [0.6886, 0.0740, 0.8665],
        [0.1366, 0.1025, 0.1841],
        [0.7264, 0.3153, 0.6871]]), tensor([[0.0756, 0.1966, 0.3164],
        [0.4017, 0.1186, 0.8274],
        [0.3821, 0.6605, 0.8536],
        [0.5932, 0.6367, 0.9826]]), tensor([[0.2745, 0.6584, 0.2775],
        [0.8573, 0.8993, 0.0390],
        [0.9268, 0.7388, 0.7179],
        [0.7058, 0.9156, 0.4340]]), tensor([[0.0772, 0.3565, 0.1479],
        [0.5331, 0.4066, 0.2318],
        [0.4545, 0.9737, 0.4606],
        [0.5159, 0.4220, 0.5786]])]
tensor([[0.2961, 0.5166, 0.2517],
        [0.6886, 0.0740, 0.8665],
        [0.1366, 0.1025, 0.1841],
        [0.7264, 0.3153, 0.6871],
        [0.0756, 0.1966, 0.3164],
        [0.4017, 0.1186, 0.8274],
        [0.3821, 0.6605, 0.8536],
        [0.5932, 0.6367, 0.9826],
        [0.2745, 0.6584, 0.2775],
        [0.8573, 0.8993, 0.0390],
        [0.9268, 0.7388, 0.7179],
        [0.7058, 0.9156, 0.4340],
        [0.0772, 0.3565, 0.1479],
     

### Demonstrate tensor.view & toensor.contiguous

In [ ]:
torch.manual_seed(123)
t1 = torch.rand(4, 6)
print(t1)
t1.view(4, 3, 2)  # unroll last dim

tensor([[0.2961, 0.5166, 0.2517, 0.6886, 0.0740, 0.8665],
        [0.1366, 0.1025, 0.1841, 0.7264, 0.3153, 0.6871],
        [0.0756, 0.1966, 0.3164, 0.4017, 0.1186, 0.8274],
        [0.3821, 0.6605, 0.8536, 0.5932, 0.6367, 0.9826]])


tensor([[[0.2961, 0.5166],
         [0.2517, 0.6886],
         [0.0740, 0.8665]],

        [[0.1366, 0.1025],
         [0.1841, 0.7264],
         [0.3153, 0.6871]],

        [[0.0756, 0.1966],
         [0.3164, 0.4017],
         [0.1186, 0.8274]],

        [[0.3821, 0.6605],
         [0.8536, 0.5932],
         [0.6367, 0.9826]]])

In [ ]:
t1.contiguous().view(4, 6)  # combine

tensor([[0.2961, 0.5166, 0.2517, 0.6886, 0.0740, 0.8665],
        [0.1366, 0.1025, 0.1841, 0.7264, 0.3153, 0.6871],
        [0.0756, 0.1966, 0.3164, 0.4017, 0.1186, 0.8274],
        [0.3821, 0.6605, 0.8536, 0.5932, 0.6367, 0.9826]])

### Demonstrate batched matrix multiplication

In [ ]:
# (b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)
a = torch.tensor(
    [
        [
            [
                [0.2745, 0.6584, 0.2775, 0.8573],
                [0.8993, 0.0390, 0.9268, 0.7388],
                [0.7179, 0.7058, 0.9156, 0.4340],
            ],
            [
                [0.0772, 0.3565, 0.1479, 0.5331],
                [0.4066, 0.2318, 0.4545, 0.9737],
                [0.4606, 0.5159, 0.4220, 0.5786],
            ],
        ]
    ]
)

a.transpose(2, 3)

tensor([[[[0.2745, 0.8993, 0.7179],
          [0.6584, 0.0390, 0.7058],
          [0.2775, 0.9268, 0.9156],
          [0.8573, 0.7388, 0.4340]],

         [[0.0772, 0.4066, 0.4606],
          [0.3565, 0.2318, 0.5159],
          [0.1479, 0.4545, 0.4220],
          [0.5331, 0.9737, 0.5786]]]])

In [285]:
print(a @ a.transpose(2, 3))

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


In this case, the matrix multiplication implementation in PyTorch will handle the 4-dimensional input tensor so that the matrix multiplication is carried out between the 2 last dimensions (num_tokens, head_dim) and then repeated for the individual heads

In [286]:
# same as above
first_head = a[0, 0, :, :]
first_res = first_head @ first_head.T
print("First head:\n", first_res)

second_head = a[0, 1, :, :]
second_res = second_head @ second_head.T
print("\nSecond head:\n", second_res)

First head:
 tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

Second head:
 tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


### Understand transpose

think transpose as changing axis

In [ ]:
torch.manual_seed(123)

t = torch.rand(2, 6, 4)
t

tensor([[[0.2961, 0.5166, 0.2517, 0.6886],
         [0.0740, 0.8665, 0.1366, 0.1025],
         [0.1841, 0.7264, 0.3153, 0.6871],
         [0.0756, 0.1966, 0.3164, 0.4017],
         [0.1186, 0.8274, 0.3821, 0.6605],
         [0.8536, 0.5932, 0.6367, 0.9826]],

        [[0.2745, 0.6584, 0.2775, 0.8573],
         [0.8993, 0.0390, 0.9268, 0.7388],
         [0.7179, 0.7058, 0.9156, 0.4340],
         [0.0772, 0.3565, 0.1479, 0.5331],
         [0.4066, 0.2318, 0.4545, 0.9737],
         [0.4606, 0.5159, 0.4220, 0.5786]]])

In [ ]:
t.transpose(0, 1)

tensor([[[0.2961, 0.5166, 0.2517, 0.6886],
         [0.2745, 0.6584, 0.2775, 0.8573]],

        [[0.0740, 0.8665, 0.1366, 0.1025],
         [0.8993, 0.0390, 0.9268, 0.7388]],

        [[0.1841, 0.7264, 0.3153, 0.6871],
         [0.7179, 0.7058, 0.9156, 0.4340]],

        [[0.0756, 0.1966, 0.3164, 0.4017],
         [0.0772, 0.3565, 0.1479, 0.5331]],

        [[0.1186, 0.8274, 0.3821, 0.6605],
         [0.4066, 0.2318, 0.4545, 0.9737]],

        [[0.8536, 0.5932, 0.6367, 0.9826],
         [0.4606, 0.5159, 0.4220, 0.5786]]])

In [ ]:
t.transpose(0, 2)

tensor([[[0.2961, 0.2745],
         [0.0740, 0.8993],
         [0.1841, 0.7179],
         [0.0756, 0.0772],
         [0.1186, 0.4066],
         [0.8536, 0.4606]],

        [[0.5166, 0.6584],
         [0.8665, 0.0390],
         [0.7264, 0.7058],
         [0.1966, 0.3565],
         [0.8274, 0.2318],
         [0.5932, 0.5159]],

        [[0.2517, 0.2775],
         [0.1366, 0.9268],
         [0.3153, 0.9156],
         [0.3164, 0.1479],
         [0.3821, 0.4545],
         [0.6367, 0.4220]],

        [[0.6886, 0.8573],
         [0.1025, 0.7388],
         [0.6871, 0.4340],
         [0.4017, 0.5331],
         [0.6605, 0.9737],
         [0.9826, 0.5786]]])